# Ventilator pressure prediction — baseline v001

A cell-by-cell version of the accepted LightGBM baseline. The feature builder,
grouped split, model settings, metric, and prediction checks match
`train_baseline.py`. The script and original artifacts remain available.

**Default: review the existing run.** Run All reads the saved metrics, training
curve, per-setting errors, prediction CSV, and Gym receipt. It does not retrain
or submit. Set `RUN_TRAINING = True` to reproduce the pipeline in a **new**
output directory; submission has a separate, disabled switch.

The original accepted result is Gym MAE **1.02635** and held-out inspiratory
MAE **1.01089**. These historical values are loaded from saved artifacts below,
not presented as a fresh notebook training run.

Use only the assignment-approved Gym data. No original Kaggle data, external
competition solutions, or pretrained competition models are used.

## 1. Run mode and paths

Open this notebook from the competition directory. On DataHub, use the project
Python environment `/home/v2gao/dsc102/pa3/.venv/bin/python` for training.
`requirements-baseline.txt` records the original baseline dependency versions.

Keep the original `outputs/baseline_v001/` unchanged. If you reproduce training,
choose a new `RUN_NAME`; an existing output directory will be refused.

In [1]:
from __future__ import annotations

import hashlib
import importlib.metadata as metadata
import json
import re
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

RUN_TRAINING = False
SUBMIT_TO_GYM = False
RUN_NAME = "baseline_v001_notebook"

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "data").is_dir():
    candidate = PROJECT_DIR / "ventilator-pressure-prediction"
    if (candidate / "data").is_dir():
        PROJECT_DIR = candidate

DATA_DIR = PROJECT_DIR / "data"
EXISTING_RUN_DIR = PROJECT_DIR / "outputs" / "baseline_v001"
args = SimpleNamespace(
    data_dir=DATA_DIR,
    output_dir=PROJECT_DIR / "outputs" / RUN_NAME,
    max_train_breaths=20000,
    trees=300,
    threads=4,
    seed=20261102,
)
ACTIVE_RUN_DIR = args.output_dir if RUN_TRAINING else EXISTING_RUN_DIR
assert DATA_DIR.is_dir(), f"Set PROJECT_DIR to the folder containing data/: {PROJECT_DIR}"
print(f"Mode: {'new training run' if RUN_TRAINING else 'review saved baseline'}")
print(f"Project: {PROJECT_DIR}")
print(f"Artifacts: {ACTIVE_RUN_DIR}")

Mode: review saved baseline
Project: /Users/generalclasher/Documents/dsc102/pa3/ventilator-pressure-prediction
Artifacts: /Users/generalclasher/Documents/dsc102/pa3/ventilator-pressure-prediction/outputs/baseline_v001


## 2. Dependencies

Review mode needs NumPy and pandas. Training additionally needs the pinned
LightGBM and scikit-learn environment. If a fresh environment needs installation,
run `%pip install -r requirements-baseline.txt` separately; this notebook does
not install or upgrade packages automatically.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

if RUN_TRAINING:
    import lightgbm as lgb
    from sklearn.model_selection import train_test_split
    print({package: metadata.version(package) for package in
           ("lightgbm", "scikit-learn", "numpy", "pandas")})
else:
    print("Review dependencies loaded; no training libraries or model are executed.")

Review dependencies loaded; no training libraries or model are executed.


## 3. Breath checks and control-history features

Each approved breath has 80 contiguous, ordered rows. The checks below enforce
that structure before reshaping. Pressure and identifiers are excluded from the
feature matrix. Lags, changes, rolling means, and cumulative features reset at
each breath. The control integral is a history proxy, not measured lung volume.

These helper functions are copied directly from the baseline script.

In [3]:
STEPS = 80
DTYPES = {
    "id": "int64", "breath_id": "int32", "R": "int8", "C": "int8",
    "time_step": "float32", "u_in": "float32", "u_out": "int8",
    "pressure": "float32",
}


def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def features(frame: pd.DataFrame) -> tuple[np.ndarray, list[str], np.ndarray]:
    """Return features in original row order, after verifying 80-row blocks."""
    if len(frame) % STEPS:
        raise ValueError("Unexpected breath length; do not reshape this file")
    blocks = frame["breath_id"].to_numpy().reshape(-1, STEPS)
    breath_ids = blocks[:, 0]
    if not np.all(blocks == breath_ids[:, None]):
        raise ValueError("Breaths are not contiguous 80-row blocks")
    if len(np.unique(breath_ids)) != len(breath_ids):
        raise ValueError("Repeated breath blocks")
    if not frame["id"].is_unique:
        raise ValueError("Duplicate row IDs")
    u = frame["u_in"].to_numpy().reshape(-1, STEPS)
    t = frame["time_step"].to_numpy().reshape(-1, STEPS)
    if not np.all(np.diff(t, axis=1) > 0):
        raise ValueError("Timestamps must increase inside each breath")
    for column in ("R", "C"):
        values = frame[column].to_numpy().reshape(-1, STEPS)
        if not np.all(values == values[:, :1]):
            raise ValueError(f"{column} changes within a breath")

    columns = {}
    for column in ("R", "C", "time_step", "u_in", "u_out"):
        columns[column] = frame[column].to_numpy(dtype=np.float32)
    step = np.arange(STEPS)
    dt = np.diff(t, axis=1, prepend=t[:, :1])
    cumulative = np.cumsum(u, axis=1, dtype=np.float32)
    columns["step"] = np.broadcast_to(step, u.shape).ravel()
    columns["dt"] = dt.ravel()
    columns["control_sum"] = cumulative.ravel()
    columns["control_mean"] = (cumulative / (step + 1)).ravel()
    # Valve opening is not measured flow: this is a history proxy, not volume.
    columns["control_integral"] = np.cumsum(u * dt, axis=1).ravel()
    for lag in (1, 2, 4, 8):
        shifted = np.empty_like(u)
        shifted[:, :lag] = u[:, :1]
        shifted[:, lag:] = u[:, :-lag]
        columns[f"u_in_lag_{lag}"] = shifted.ravel()
        columns[f"u_in_change_{lag}"] = (u - shifted).ravel()
    padded_sum = np.concatenate([np.zeros((len(u), 1), dtype=np.float32), cumulative], axis=1)
    for window in (4, 8):
        begin = np.maximum(0, step + 1 - window)
        columns[f"u_in_mean_{window}"] = (
            (padded_sum[:, step + 1] - padded_sum[:, begin]) / (step + 1 - begin)
        ).ravel()
    names = list(columns)
    matrix = np.column_stack(list(columns.values())).astype(np.float32, copy=False)
    if not np.isfinite(matrix).all() or "pressure" in names:
        raise ValueError("Invalid features")
    return matrix, names, breath_ids

## 4. Load the approved data and build features

Training mode records source-file hashes and builds the matrix. Review mode
skips this allocation and later reads the existing run's saved artifacts.

In [4]:
if RUN_TRAINING:
    args.output_dir.mkdir(parents=True, exist_ok=False)
    started = time.monotonic()
    hashes = {name: digest(args.data_dir / name) for name in ("train.csv", "test.csv", "sample_submission.csv")}
    print("Loading approved training data", flush=True)
    train = pd.read_csv(args.data_dir / "train.csv", dtype=DTYPES)
    x, names, breath_ids = features(train)
    settings = train[["R", "C"]].to_numpy()[::STEPS]
    strata = np.array([f"{r}_{c}" for r, c in settings])
else:
    print("Review mode: training cells are skipped; saved baseline results are shown below.")

Review mode: training cells are skipped; saved baseline results are shown below.


## 5. Hold out whole breaths

The validation set is 10% of whole breaths, stratified by R/C combination.
The first model fits 20,000 breaths from the remaining pool. Validation and
fitting both use only inspiration points (`u_out == 0`). The split is fixed by
seed; no pressure labels are used to construct features or select groups.

In [5]:
if RUN_TRAINING:
    pool, valid_groups = train_test_split(
        np.arange(len(breath_ids)), test_size=0.1, stratify=strata,
        random_state=args.seed,
    )
    if 0 < args.max_train_breaths < len(pool):
        used_groups, _ = train_test_split(
            pool, train_size=args.max_train_breaths, stratify=strata[pool],
            random_state=args.seed + 1,
        )
    else:
        used_groups = pool
    if np.intersect1d(breath_ids[used_groups], breath_ids[valid_groups]).size:
        raise ValueError("Training and validation breath leakage")
    train_blocks = np.zeros(len(breath_ids), dtype=bool)
    valid_blocks = np.zeros(len(breath_ids), dtype=bool)
    train_blocks[used_groups] = True
    valid_blocks[valid_groups] = True
    inspiration = train["u_out"].to_numpy() == 0
    train_mask = np.repeat(train_blocks, STEPS) & inspiration
    valid_mask = np.repeat(valid_blocks, STEPS) & inspiration
    y = train["pressure"].to_numpy()
    x_train, y_train = x[train_mask], y[train_mask]
    x_valid, y_valid = x[valid_mask], y[valid_mask]
    print(f"Training: {len(used_groups)} breaths / {len(y_train)} scored rows", flush=True)
    print(f"Validation: {len(valid_groups)} breaths / {len(y_valid)} scored rows", flush=True)

## 6. Fit the first LightGBM model

Same baseline settings: L1 objective, 300-tree cap, 63 leaves, learning rate
0.07, four threads, and early stopping on held-out inspiratory MAE.
During an enabled training run, iteration results appear under this cell.

In [6]:
if RUN_TRAINING:
    model = lgb.LGBMRegressor(
        objective="regression_l1", n_estimators=args.trees, learning_rate=0.07,
        num_leaves=63, min_child_samples=80, max_bin=127,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=1.0,
        n_jobs=args.threads, random_state=args.seed, deterministic=True,
        force_col_wise=True, verbosity=-1,
    )
    fit_start = time.monotonic()
    model.fit(
        x_train, y_train, feature_name=names,
        eval_set=[(x_valid, y_valid)], eval_metric="l1",
        callbacks=[lgb.early_stopping(30, first_metric_only=True), lgb.log_evaluation(25)],
    )
    fit_seconds = time.monotonic() - fit_start

## 7. Validate and save the checkpoint

MAE pools absolute errors over every scored validation point. Per-R/C errors,
the model, and exact trained/validation breath IDs are saved for tracking.

In [7]:
if RUN_TRAINING:
    valid_prediction = model.predict(x_valid, num_iteration=model.best_iteration_)
    error = np.abs(valid_prediction - y_valid)
    validation_mae = float(error.sum(dtype=np.float64) / len(error))
    slices = train.loc[valid_mask, ["R", "C"]].copy()
    slices["absolute_error"] = error
    by_rc = slices.groupby(["R", "C"])["absolute_error"].agg(["mean", "count"])
    by_rc.to_csv(args.output_dir / "validation_by_rc.csv")
    print(f"Inspiratory validation MAE: {validation_mae:.6f}", flush=True)
    model.booster_.save_model(str(args.output_dir / "model.txt"), num_iteration=model.best_iteration_)
    np.savez_compressed(
        args.output_dir / "split_breath_ids.npz",
        trained=breath_ids[used_groups], validation=breath_ids[valid_groups],
    )

## 8. Generate and verify the prediction CSV

Predict every test row, including unscored expiration rows, and explicitly
restore sample-submission ID order. Re-read the written CSV to check its schema,
ID order, and finite values before it could be submitted.

In [8]:
if RUN_TRAINING:
    # Training feature memory can be released before constructing the test matrix.
    del x, x_train, x_valid, train, slices
    test = pd.read_csv(args.data_dir / "test.csv", dtype=DTYPES)
    x_test, test_names, test_breath_ids = features(test)
    if test_names != names or np.intersect1d(breath_ids, test_breath_ids).size:
        raise ValueError("Unexpected test schema or overlapping breath IDs")
    prediction = model.predict(x_test, num_iteration=model.best_iteration_)
    sample = pd.read_csv(args.data_dir / "sample_submission.csv")
    if not sample["id"].is_unique or len(sample) != len(test):
        raise ValueError("Invalid sample submission IDs/count")
    ordered = pd.Series(prediction, index=test["id"].to_numpy()).reindex(sample["id"])
    if not np.isfinite(ordered.to_numpy()).all():
        raise ValueError("Missing or nonfinite prediction, or ID mismatch")
    submission = pd.DataFrame({"id": sample["id"].to_numpy(), "pressure": ordered.to_numpy()})
    submission_path = args.output_dir / "submission.csv"
    submission.to_csv(submission_path, index=False, float_format="%.8f")
    # Verify the serialized artifact that will actually be submitted.
    saved = pd.read_csv(submission_path)
    assert list(saved.columns) == ["id", "pressure"]
    assert np.array_equal(saved["id"].to_numpy(), sample["id"].to_numpy())
    assert np.isfinite(saved["pressure"].to_numpy()).all()

## 9. Record the run

Save parameters, versions, hashes, row counts, validation MAE, and runtime in
`metrics.json`. New notebook runs use their new output directory; the original
accepted run and receipt are not overwritten.

In [9]:
if RUN_TRAINING:
    results = {
        "experiment": "baseline_v001", "seed": args.seed,
        "data_sha256": hashes, "features": names,
        "versions": {p: metadata.version(p) for p in ("lightgbm", "scikit-learn", "numpy", "pandas")},
        "model_parameters": model.get_params(), "best_iteration": model.best_iteration_,
        "training_breaths": len(used_groups), "validation_breaths": len(valid_groups),
        "training_scored_rows": int(train_mask.sum()), "validation_scored_rows": int(valid_mask.sum()),
        "validation_inspiratory_mae": validation_mae,
        "fit_seconds": fit_seconds, "total_seconds": time.monotonic() - started,
        "submission_rows": len(saved), "submission_sha256": digest(submission_path),
        "prediction_min": float(ordered.min()), "prediction_max": float(ordered.max()),
        "all_submission_checks_passed": True,
        "training_scope": "20000 sampled training breaths by default; no full-data refit for this first submission",
    }
    (args.output_dir / "metrics.json").write_text(json.dumps(results, indent=2) + "\n")
    print(json.dumps(results, indent=2), flush=True)

## 10. Review results and artifact checks

This section runs in either mode. In the default review mode it displays the
previously trained baseline. After a new training run it displays that run's
artifacts, with no Gym score until a new submission receipt exists.

In [10]:
metrics_path = ACTIVE_RUN_DIR / "metrics.json"
assert metrics_path.is_file(), f"No completed run at {ACTIVE_RUN_DIR}"
run_metrics = json.loads(metrics_path.read_text())
receipt_path = ACTIVE_RUN_DIR / "submission_receipt.json"
run_receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else None
summary = {
    "Artifact directory": str(ACTIVE_RUN_DIR),
    "Seed": run_metrics["seed"],
    "Training breaths": run_metrics["training_breaths"],
    "Validation breaths": run_metrics["validation_breaths"],
    "Best iteration": run_metrics["best_iteration"],
    "Inspiratory validation MAE": run_metrics["validation_inspiratory_mae"],
    "Gym MAE": run_receipt["score"] if run_receipt else "Not submitted",
    "Training seconds": run_metrics["fit_seconds"],
    "Full pipeline seconds": run_metrics["total_seconds"],
    "Submission rows": run_metrics["submission_rows"],
}
display(pd.DataFrame(summary.items(), columns=["Metric", "Value"]).set_index("Metric"))

csv_path = ACTIVE_RUN_DIR / "submission.csv"
assert digest(csv_path) == run_metrics["submission_sha256"]
if run_receipt:
    assert digest(csv_path) == run_receipt["submission_sha256"]
    assert run_receipt["accepted"]
saved_predictions = pd.read_csv(csv_path)
sample_ids = pd.read_csv(DATA_DIR / "sample_submission.csv", usecols=["id"])
assert list(saved_predictions.columns) == ["id", "pressure"]
assert saved_predictions["id"].is_unique
assert np.array_equal(saved_predictions["id"].to_numpy(), sample_ids["id"].to_numpy())
assert np.isfinite(saved_predictions["pressure"].to_numpy()).all()
print(f"PASS: {len(saved_predictions):,} predictions, exact sample ID order, finite values, matching hash")
display(saved_predictions.head())

,Value
Metric,
Artifact directory,/Users/generalclasher/Documents/dsc102/pa3/ven...
Seed,20261102
Training breaths,20000
Validation breaths,6791
Best iteration,300
Inspiratory validation MAE,1.01089
Gym MAE,1.02635
Training seconds,10.342663
Full pipeline seconds,26.136865


PASS: 603,600 predictions, exact sample ID order, finite values, matching hash


,id,pressure
0,1,5.544986
1,2,5.508813
2,3,6.265199
3,4,7.824122
4,5,11.291982


### Errors by lung setting

These are validation diagnostics, not separate Gym scores. Counts are included
because the global metric weights scored time steps, rather than giving each
R/C setting equal weight.

In [11]:
by_rc_results = pd.read_csv(ACTIVE_RUN_DIR / "validation_by_rc.csv")
display(by_rc_results.rename(columns={"mean": "Inspiratory MAE", "count": "Scored rows"}))

,R,C,Inspiratory MAE,Scored rows
0,5,10,0.899576,22461
1,5,20,0.759499,23070
2,5,50,0.432553,22061
3,20,10,1.012851,16589
4,20,20,0.807453,16704
5,20,50,0.863237,21893
6,50,10,1.518069,37444
7,50,20,1.161126,23137
8,50,50,1.239128,22739


### Original training progress

Load iteration metrics from the saved log when available. A newly trained
notebook run already shows its live progress under the fitting cell above.

In [12]:
log_candidates = [
    ACTIVE_RUN_DIR / "training.log",
    PROJECT_DIR / "runs" / f"{run_metrics['experiment']}.log",
]
if not RUN_TRAINING:
    available_log = next((path for path in log_candidates if path.is_file()), None)
    if available_log:
        entries = re.findall(r"\[(\d+)\]\s+valid_0's l1:\s+([0-9.eE+-]+)", available_log.read_text())
        curve = pd.DataFrame(entries, columns=["Iteration", "Inspiratory validation MAE"])
        if len(curve):
            curve = curve.astype({"Iteration": int, "Inspiratory validation MAE": float})
            display(curve)
    else:
        print("No saved training log found; scalar metrics remain available above.")
else:
    curve = pd.DataFrame(model.evals_result_["valid_0"])
    curve.index = pd.RangeIndex(1, len(curve) + 1, name="Iteration")
    display(curve.rename(columns={"l1": "Inspiratory validation MAE"}))

,Iteration,Inspiratory validation MAE
0,25,2.31278
1,50,1.53800
2,75,1.32907
3,100,1.21752
4,125,1.15360
5,150,1.11486
6,175,1.09063
7,200,1.06992
8,225,1.05300
9,250,1.03936


## 11. Optional Gym submission — disabled by default

Set `SUBMIT_TO_GYM = True` only when intentionally submitting a newly trained
notebook run. Review mode cannot upload the already-accepted baseline. The
upload command disables optional AI session-log upload and saves its receipt.
Re-running this cell refuses an existing receipt/log to prevent a duplicate.

In [13]:
if SUBMIT_TO_GYM:
    if not RUN_TRAINING:
        raise RuntimeError("Review mode does not resubmit the accepted baseline. Use a new training run.")
    cli = Path(sys.executable).with_name("aicodinggym")
    if not cli.is_file():
        raise RuntimeError("Select the configured project virtual-environment kernel before submission.")
    log_path = ACTIVE_RUN_DIR / "notebook-submit.log"
    if receipt_path.exists() or log_path.exists():
        raise FileExistsError("This run already has a submission record; no upload was performed.")
    command = [
        str(cli), "mle", "submit", "ventilator-pressure-prediction",
        "-F", str(csv_path), "--workspace-dir", str(PROJECT_DIR.parent),
        "--no-upload-logs", "-m", f"{RUN_NAME}: notebook reproduction of baseline v001",
    ]
    print(shlex.join(command))
    with log_path.open("x") as output:
        completed = subprocess.run(command, cwd=PROJECT_DIR, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output.write(completed.stdout)
    print(completed.stdout)
    completed.check_returncode()
    score_match = re.search(r"Score:\s*([0-9.eE+-]+)", completed.stdout)
    accepted = "Successfully submitted prediction" in completed.stdout
    if not accepted or not score_match:
        raise RuntimeError("Inspect the saved CLI response; an accepted numeric score was not confirmed.")
    new_receipt = {
        "platform": "AI Coding Gym", "competition_id": "ventilator-pressure-prediction",
        "accepted": True, "score": float(score_match.group(1)),
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "submission_path": str(csv_path), "submission_sha256": digest(csv_path),
        "ai_session_log_upload": False, "cli_receipt": completed.stdout,
    }
    with receipt_path.open("x") as output:
        json.dump(new_receipt, output, indent=2)
        output.write("\n")
    print(f"Accepted Gym MAE: {new_receipt['score']}")
else:
    print("Submission disabled. Existing Gym result and artifacts are preserved.")

Submission disabled. Existing Gym result and artifacts are preserved.


## Ideas for the next experiment

- Raise the tree cap while keeping the split and training sample fixed; the
  original validation curve was still improving at 300 trees.
- Increase the number of training breaths as a separate change.
- Compare the proposed compact GRU on the same split and inspiratory metric.
- Investigate the larger errors in high-resistance settings and confirm
  promising changes on another grouped split/seed.

These are proposals, not additional completed experiments. See `EXPERIMENTS.md`
for the first run's record. The public benchmark source provides scores but
not implementation traces, so exact bugs in those agents remain unverified.